# Lezione 11 — Ricerca e recupero classico

**Corso:** NLP Classico con Python · **Blocco 4:** Significato e ricerca
**Tempo stimato:** 105 minuti · **Capitolo del manuale:** `manual/11_busqueda_recuperacion_clasica_IT.md`

## Cosa vedrai in questa lezione

- Il **recupero dell'informazione** come problema di ordinare documenti per rilevanza rispetto a una query.
- Un motore di ricerca con **TF-IDF e similarità del coseno**, programmato con NumPy, su un piccolo esempio calcolato a mano.
- **BM25** (`k1`, `b`), anch'esso programmato con NumPy: perché il suo IDF probabilistico può essere **negativo** quando un termine compare in più della metà dei documenti, e come si corregge.
- Valutazione con **precisione@k** e **MRR** su una collezione di domande frequenti (FAQ) con rilevanza fissata in anticipo.
- Un caso concreto in cui la ricerca per parole **fallisce** davanti a sinonimi e parafrasi: la motivazione per gli embedding densi come recuperatore alternativo.

**Requisiti:** lezione 4 (TF-IDF, sacco di parole); l'intuizione del coseno della lezione 10 aiuta ma non è indispensabile. **Librerie:** `numpy`, `pandas`, `nltk` (solo per le liste di stopword). **Dati:** `datos/faq_es.csv`, `datos/faq_it.csv` (collezione di FAQ) e `datos/faq_consultas_es.csv`, `datos/faq_consultas_it.csv` (query con rilevanza fissata in anticipo); in Colab, carica la cartella `datos/`. Tutte le cifre sono un *esempio illustrativo*, calcolate su una collezione sintetica di 40 documenti per lingua. **Durata dell'esecuzione:** pochi secondi.

---
## Cella 0 — Setup

In [ ]:
import subprocess, sys

def asegurar(paquete, modulo):
    """Installa il pacchetto solo se manca (idempotente e silenzioso)."""
    try:
        __import__(modulo)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)

asegurar("numpy", "numpy")
asegurar("pandas", "pandas")
asegurar("matplotlib", "matplotlib")
print("Librerie pronte.")

In [ ]:
import os
import re
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
np.random.seed(42)                        # seme fisso (anche se questo argomento non addestra nulla di casuale)
plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.grid"] = True

try:
    import nltk
    from nltk.corpus import stopwords
    stopwords.words("spanish")
except LookupError:
    nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords
STOPWORDS = {"es": set(stopwords.words("spanish")), "it": set(stopwords.words("italian"))}   # dato-es

def cargar_datos(nombre):
    """Cerca un file di dati in datos/, ../datos/ e /content/datos/."""
    for base in ("datos", "../datos", "/content/datos"):
        ruta = os.path.join(base, nombre)
        if os.path.exists(ruta):
            return ruta
    # Copia pubblica (Colab): se non c'è in locale, scarica il file dal repository dei notebook.
    try:
        import urllib.request
        destino = os.path.join("datos", nombre)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/RaffaeleBini/arcadia-notebooks/main/14-nlp-classico/datos/{nombre}", destino)
        return destino
    except Exception:
        pass
    raise FileNotFoundError(
        "Non trovo " + nombre + ". Carica la cartella datos/ del corso (in Colab: pannello dei file) o lasciala accanto al notebook.")

def tokenizar(texto, idioma, quitar_stopwords=True):
    """Minuscole, solo lettere (senza numeri né punteggiatura), e stopword opzionali."""
    palabras = re.findall(r"[^\W\d_]+", texto.lower())
    if quitar_stopwords:
        palabras = [p for p in palabras if p not in STOPWORDS[idioma]]
    return palabras

def construir_vocabulario(documentos_tok):
    """Lista ordinata di parole distinte che compaiono in almeno un documento."""
    vocab = sorted({p for doc in documentos_tok for p in doc})
    return vocab, {p: i for i, p in enumerate(vocab)}

def matriz_tf(documentos_tok, indice):
    """Matrice di conteggi (documenti x vocabolario): TF grezzo, non normalizzato."""
    M = np.zeros((len(documentos_tok), len(indice)), dtype=float)
    for i, doc in enumerate(documentos_tok):
        for palabra, n in Counter(doc).items():
            if palabra in indice:
                M[i, indice[palabra]] = n
    return M

def idf_tfidf(TF):
    """IDF classico e con smoothing: log((1+N)/(1+df)) + 1, sempre >= 1 (mai negativo)."""
    N = TF.shape[0]
    df = (TF > 0).sum(axis=0)
    return np.log((1 + N) / (1 + df)) + 1.0

def matriz_tfidf(TF, idf):
    """TF-IDF = TF * IDF, con ogni riga normalizzata a norma 1 (così il coseno è un prodotto scalare)."""
    X = TF * idf
    normas = np.linalg.norm(X, axis=1, keepdims=True)
    normas[normas == 0] = 1.0
    return X / normas

def vector_consulta(consulta_tok, indice, idf):
    """Stesso vocabolario e IDF dei documenti; TF della query = conteggio di ogni parola."""
    v = np.zeros(len(indice))
    for palabra, n in Counter(consulta_tok).items():
        if palabra in indice:
            v[indice[palabra]] = n
    v = v * idf
    norma = np.linalg.norm(v)
    return v / norma if norma > 0 else v

def buscar_coseno(consulta_tok, X, indice, idf, k=5):
    """Ordina i documenti per coseno con la query; restituisce (indici, punteggi)."""
    q = vector_consulta(consulta_tok, indice, idf)
    puntuaciones = X @ q
    orden = np.argsort(-puntuaciones)[:k]
    return orden, puntuaciones[orden]

def idf_bm25(TF, corregido=False):
    """IDF probabilistico di Robertson-Sparck Jones: log((N-df+0.5)/(df+0.5)); può essere NEGATIVO se df > N/2. Con corregido=True, log(1 + (N-df+0.5)/(df+0.5)), sempre >= 0."""
    N = TF.shape[0]
    df = (TF > 0).sum(axis=0)
    razon = (N - df + 0.5) / (df + 0.5)
    return np.log(1.0 + razon) if corregido else np.log(razon)

def bm25_matriz(TF, idf, k1=1.5, b=0.75):
    """Matrice BM25 (documenti x vocabolario): per ogni parola e documento, il termine che si somma al punteggio se la parola è nella query."""
    longitudes = TF.sum(axis=1, keepdims=True)
    long_media = longitudes.mean()
    denom = TF + k1 * (1 - b + b * longitudes / long_media)
    return idf * (TF * (k1 + 1) / np.where(denom == 0, 1, denom))

def buscar_bm25(consulta_tok, BM25, indice, k=5):
    """Somma, per ogni documento, la colonna BM25 di ogni parola della query presente nel vocabolario."""
    cols = [indice[p] for p in consulta_tok if p in indice]
    puntuaciones = BM25[:, cols].sum(axis=1) if cols else np.zeros(BM25.shape[0])
    orden = np.argsort(-puntuaciones)[:k]
    return orden, puntuaciones[orden]

def precision_en_k(recuperados, relevantes, k):
    """Frazione di rilevanti tra i primi k recuperati."""
    top = recuperados[:k]
    return len([d for d in top if d in relevantes]) / k

def rango_reciproco(recuperados, relevantes):
    """1 / posizione (indicizzata da 1) del primo documento rilevante; 0 se non compare."""
    for i, d in enumerate(recuperados, 1):
        if d in relevantes:
            return 1.0 / i
    return 0.0

def palabras_clave(texto, idioma, TF_global, idf_global, indice_global, n=5):
    """Le n parole del documento con il TF-IDF più alto (TF locale del documento, IDF di tutto il corpus)."""
    tok = tokenizar(texto, idioma)
    cuenta = Counter(tok)
    puntuadas = [(p, n_ * idf_global[indice_global[p]]) for p, n_ in cuenta.items() if p in indice_global]
    puntuadas.sort(key=lambda par: -par[1])
    return [p for p, _ in puntuadas[:n]]

def dividir_frases(texto):
    """Taglia un testo in frasi per . ! ? e scarta quelle vuote."""
    return [f.strip() for f in re.split(r"(?<=[.!?])\s+", texto.strip()) if f.strip()]

def resumen_extractivo(texto, idioma, idf_global, indice_global, n_frases=2):
    """Punteggia ogni frase con la media del TF-IDF delle sue parole (TF nella frase) e restituisce le n_frases migliori, nel loro ordine originale."""
    frases = dividir_frases(texto)
    puntuaciones = []
    for frase in frases:
        tok = tokenizar(frase, idioma)
        if not tok:
            puntuaciones.append(0.0)
            continue
        cuenta = Counter(tok)
        pesos = [cuenta[p] * idf_global[indice_global[p]] for p in cuenta if p in indice_global]
        puntuaciones.append(float(np.mean(pesos)) if pesos else 0.0)
    mejores = sorted(range(len(frases)), key=lambda i: -puntuaciones[i])[:n_frases]
    mejores.sort()
    return [frases[i] for i in mejores], puntuaciones

print("Funzioni ausiliarie caricate.")

---
## 1. TF-IDF e coseno: motore di ricerca a mano

Quattro documenti minuscoli (*esempio illustrativo*) per calcolare il TF-IDF a mano e verificarlo con il codice.

In [ ]:
# datos-es
# Documento minuscolo per calcolare TF-IDF e BM25 a mano (identico in ES e IT: sono dati)
TOY = ["el gato come pescado", "el gato come pescado pescado pescado", "el perro corre mucho", "el pescado nada"]

In [ ]:
toy_tok = [t.split() for t in TOY]                       # tokenizzazione banale: sono già in minuscolo e senza punteggiatura
vocab_t, idx_t = construir_vocabulario(toy_tok)
TFt = matriz_tf(toy_tok, idx_t)
idf_t = idf_tfidf(TFt)
print("vocabolario:", vocab_t)
print("df:", {w: int((TFt[:, idx_t[w]] > 0).sum()) for w in vocab_t})
print("idf(pescado) =", round(float(idf_t[idx_t["pescado"]]), 3), "| idf(gato) =", round(float(idf_t[idx_t["gato"]]), 3))
assert vocab_t.count("pescado") == 1 and int((TFt[:, idx_t["pescado"]] > 0).sum()) == 3
assert np.isclose(idf_t[idx_t["pescado"]], np.log(5 / 4) + 1)          # calcolo a mano

### Cosa osservi?

`pescado` compare in 3 dei 4 documenti: `idf = log(5/4) + 1 ≈ 1,223`, esattamente come calcolato a mano nel capitolo. L'IDF di questa formula (quella che usa `TfidfVectorizer` di default) **non è mai negativo**: al massimo vale 1 quando la parola è in tutti i documenti.

In [ ]:
X_t = matriz_tfidf(TFt, idf_t)
q_tok = tokenizar("pescado", "es", quitar_stopwords=False)   # dato-es
orden, punt = buscar_coseno(q_tok, X_t, idx_t, idf_t, k=4)
for i, p in zip(orden, punt):
    print(f"  {TOY[i]!r:45} -> {p:.3f}")
assert list(orden) == [1, 3, 0, 2]     # il doc che ripete di più "pescado" esce primo; l'unico senza quella parola risulta ultimo

---
## 2. BM25: la formula, e il problema dell'IDF negativo

Con gli stessi 4 documenti, l'IDF probabilistico di BM25 (`log((N-df+0,5)/(df+0,5))`) per `pescado` (`df=3` su `N=4`, più della metà) e per `el` (`df=4`, tutti i documenti):

In [ ]:
idf_b = idf_bm25(TFt)
idf_b_corr = idf_bm25(TFt, corregido=True)
palabras_toy = ("pescado", "el", "corre")   # dato-es
for w in palabras_toy:
    print(f"  {w:>8} df={int((TFt[:, idx_t[w]] > 0).sum())}  idf_bm25={idf_b[idx_t[w]]:.3f}  idf_bm25_corregido={idf_b_corr[idx_t[w]]:.3f}")
palabra_el = "el"   # dato-es
assert idf_b[idx_t["pescado"]] < 0 and idf_b[idx_t[palabra_el]] < 0        # df > N/2 -> idf negativo
assert idf_b_corr[idx_t["pescado"]] > 0 and idf_b_corr[idx_t[palabra_el]] > 0

### Cosa osservi?

`el` è in tutti e 4 i documenti: `idf_bm25 = log(0,5/4,5) = -2,197`. `pescado` è in 3 su 4: `idf_bm25 = log(1,5/3,5) = -0,847`. Entrambi negativi, come spiegato nel capitolo (sezione 3.1): non appena `df > N/2`, il logaritmo di un numero minore di 1 è negativo. `corre` (`df=1`) dà un `idf_bm25` positivo. Con l'IDF corretto (`log(1+...)`), tutti e tre sono positivi.

**Il problema in pratica:** con `idf` negativo, il punteggio BM25 del termine `pescado` in ogni documento risulta invertito.

In [ ]:
k1, b = 1.5, 0.75
BM25_malo = bm25_matriz(TFt, idf_b, k1=k1, b=b)
BM25_bueno = bm25_matriz(TFt, idf_b_corr, k1=k1, b=b)
i_p = idx_t["pescado"]
print("lunghezze dei documenti:", TFt.sum(axis=1).astype(int), "| media:", round(float(TFt.sum(axis=1).mean()), 2))
print("BM25(pescado) con idf NON corretto:", np.round(BM25_malo[:, i_p], 3))
print("BM25(pescado) con idf corretto:      ", np.round(BM25_bueno[:, i_p], 3))
orden_malo = np.argsort(-BM25_malo[:, i_p])
orden_bueno = np.argsort(-BM25_bueno[:, i_p])
print("ordine NON corretto, dal peggiore al migliore:", TOY[orden_malo[-1]], "...", TOY[orden_malo[0]], "(primo)")
print("ordine corretto, primo:", TOY[orden_bueno[0]])
assert orden_malo[0] == 2 and BM25_malo[2, i_p] == 0.0     # il doc SENZA "pescado" risulta primo: è sbagliato
assert orden_bueno[0] == 1                                  # con l'idf corretto, il doc che ripete di più "pescado" risulta primo
assert BM25_malo[1, i_p] < BM25_malo[0, i_p] < BM25_malo[2, i_p]   # ordine invertito con idf non corretto

### Cosa osservi?

Con l'IDF non corretto, il documento che **non** contiene `pescado` (`el perro corre mucho`) ottiene punteggio 0 e risulta **primo**, davanti ai tre documenti che invece lo menzionano; quello che lo ripete di più risulta ultimo. Con l'IDF corretto, l'ordine si sistema: il documento che ripete di più `pescado` (aggiustato per la sua lunghezza) risulta primo. È esattamente la tabella della sezione 3 del capitolo.

---
## 3. E in un corpus reale? Stopword e l'IDF negativo

Carichiamo la collezione di FAQ (40 documenti per lingua) e verifichiamo quante parole hanno `idf_bm25` negativo, **con e senza togliere le stopword** durante la tokenizzazione.

In [ ]:
IDIOMAS = ("es", "it")   # dato-es
FAQ, TOK, VOCAB, IDX_C, TF_C, IDF_TFIDF, X_C, IDF_BM25, BM25_C = {}, {}, {}, {}, {}, {}, {}, {}, {}
for idioma in IDIOMAS:
    FAQ[idioma] = pd.read_csv(cargar_datos(f"faq_{idioma}.csv"))
    docs = (FAQ[idioma]["titulo"] + ". " + FAQ[idioma]["texto"]).tolist()
    tok_sin = [tokenizar(d, idioma, quitar_stopwords=False) for d in docs]
    vocab_sin, idx_sin = construir_vocabulario(tok_sin)
    TF_sin = matriz_tf(tok_sin, idx_sin)
    idf_sin = idf_bm25(TF_sin)
    negativas_sin = int((idf_sin < 0).sum())
    TOK[idioma] = [tokenizar(d, idioma) for d in docs]                 # con le stopword tolte: la pipeline reale del capitolo
    VOCAB[idioma], IDX_C[idioma] = construir_vocabulario(TOK[idioma])
    TF_C[idioma] = matriz_tf(TOK[idioma], IDX_C[idioma])
    IDF_TFIDF[idioma] = idf_tfidf(TF_C[idioma])
    X_C[idioma] = matriz_tfidf(TF_C[idioma], IDF_TFIDF[idioma])
    IDF_BM25[idioma] = idf_bm25(TF_C[idioma])
    BM25_C[idioma] = bm25_matriz(TF_C[idioma], IDF_BM25[idioma])
    negativas_con = int((IDF_BM25[idioma] < 0).sum())
    print(idioma, "| doc:", len(FAQ[idioma]), "| vocabolario:", len(VOCAB[idioma]),
          "| parole con idf_bm25 < 0 SENZA togliere le stopword:", negativas_sin, "su", len(vocab_sin),
          "| CON le stopword tolte:", negativas_con, "su", len(VOCAB[idioma]))
    assert negativas_sin > 0 and negativas_con == 0     # il risultato del capitolo, sezione 4

### Cosa osservi?

Senza togliere le stopword, diverse parole superano `df > N/2 = 20` (`de`, `y`, `la`… in spagnolo; `e`, `di`, `in`… in italiano) e il loro `idf_bm25` è negativo. Togliendo le stopword prima di costruire il vocabolario, nessuna delle 556 (spagnolo) o 584 (italiano) parole restanti arriva a superare la metà dei documenti: il problema scompare in questa collezione, anche se non c'è garanzia che accada sempre (per questo il codice di questo capitolo usa comunque l'IDF corretto, sezione 2).

---
## 4. Valutazione: precisione@k e MRR

Le query e il loro documento rilevante sono fissati in anticipo in `datos/faq_consultas_es.csv` / `_it.csv` (colonna `tipo`: `facil` o `dificil`). Cerchiamo con TF-IDF+coseno e con BM25 (IDF corretto) e calcoliamo precisione@3 e MRR, separando per tipo di query.

In [ ]:
def evaluar(idioma, k=3):
    consultas = pd.read_csv(cargar_datos(f"faq_consultas_{idioma}.csv"))
    filas = []
    for _, fila in consultas.iterrows():
        relevantes = {int(x) for x in str(fila["id_documento_relevante"]).split(";")}
        q_tok = tokenizar(fila["consulta"], idioma)
        orden_c, _ = buscar_coseno(q_tok, X_C[idioma], IDX_C[idioma], IDF_TFIDF[idioma], k=len(FAQ[idioma]))
        orden_b, _ = buscar_bm25(q_tok, BM25_C[idioma], IDX_C[idioma], k=len(FAQ[idioma]))
        ids_c = [int(FAQ[idioma].iloc[i]["id"]) for i in orden_c]
        ids_b = [int(FAQ[idioma].iloc[i]["id"]) for i in orden_b]
        filas.append(dict(tipo=fila["tipo"], p_c=precision_en_k(ids_c, relevantes, k), mrr_c=rango_reciproco(ids_c, relevantes),
                           p_b=precision_en_k(ids_b, relevantes, k), mrr_b=rango_reciproco(ids_b, relevantes)))
    return pd.DataFrame(filas)

RESULTADOS = {idioma: evaluar(idioma) for idioma in IDIOMAS}
etq_p3 = "precisione@3"
etq_todas = "tutte"
for idioma, res in RESULTADOS.items():
    print("=" * 8, idioma)
    for tipo, sub in [(etq_todas, res), ("facil", res[res.tipo == "facil"]), ("dificil", res[res.tipo == "dificil"])]:
        print(f"  {tipo:>8} (n={len(sub):2d}) | TF-IDF: {etq_p3}={sub.p_c.mean():.3f} MRR={sub.mrr_c.mean():.3f}"
              f" | BM25: {etq_p3}={sub.p_b.mean():.3f} MRR={sub.mrr_b.mean():.3f}")
    assert res[res.tipo == "facil"].mrr_c.mean() > res[res.tipo == "dificil"].mrr_c.mean()   # le facili devono andare meglio delle difficili

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(10, 4))
for eje, idioma in zip(ejes, IDIOMAS):
    res = RESULTADOS[idioma]
    medias = res.groupby("tipo")[["mrr_c", "mrr_b"]].mean()
    medias.columns = ["TF-IDF", "BM25"]
    medias.plot.bar(ax=eje, rot=0, ylim=(0, 1.05), legend=(idioma == IDIOMAS[0]))
    titulo_mrr = "MRR per tipo di query"
    eje.set_title(f"{titulo_mrr} ({idioma})")
    eje.set_ylabel("MRR")
plt.tight_layout(); plt.show()

### Cosa osservi?

Nelle query facili, il MRR è 1,0 in entrambe le lingue e con entrambi i motori: basta condividere il vocabolario. Nelle difficili scende alla metà o meno. La precisione@3 resta bassa anche nelle facili perché quasi tutte hanno un solo documento rilevante (il massimo possibile con `k=3` è 1/3). TF-IDF e BM25 danno risultati molto simili in questa collezione di documenti corti: BM25 non è automaticamente migliore.

---
## 5. Un fallimento concreto: sinonimi e parafrasi

Guardiamo da vicino due query «difficili»: una che usa un sinonimo di una parola chiave del documento, e un'altra che parafrasa l'intera frase.

In [ ]:
# datos-es
CONSULTAS_DEMO = {"es": ["cómo recupero mi contraseña olvidada", "olvidé mi clave de acceso a la cuenta", "cómo colaboro con mis compañeros de trabajo en un proyecto"],
                  "it": ["come recupero la mia password dimenticata", "ho dimenticato la chiave di accesso al mio account", "come collaboro con i miei colleghi su un progetto"]}

In [ ]:
for idioma in IDIOMAS:
    print("=" * 8, idioma)
    for consulta in CONSULTAS_DEMO[idioma]:
        q_tok = tokenizar(consulta, idioma)
        orden_c, punt_c = buscar_coseno(q_tok, X_C[idioma], IDX_C[idioma], IDF_TFIDF[idioma], k=3)
        print(f"  «{consulta}»")
        for i, p in zip(orden_c, punt_c):
            print(f"      {int(FAQ[idioma].iloc[i]['id']):3d}  {FAQ[idioma].iloc[i]['titulo']:<55} {p:.3f}")

### Cosa osservi?

In italiano, *"come collaboro con i miei colleghi su un progetto"* non trova tra i primi 3 il documento sul condividere cartelle con il team: nessuna parola coincide, il punteggio è 0 per tutti i documenti. *"ho dimenticato la chiave di accesso al mio account"* sta leggermente meglio perché condivide comunque le parole `dimenticato` e `accesso` con il documento corretto. È lo stesso fenomeno spiegato nella sezione 9 del capitolo, con le cifre complete di posizione e MRR.

---
## Riepilogo

- Il **recupero dell'informazione** ordina i documenti per rilevanza rispetto a una query; TF-IDF+coseno e BM25 sono due modi di calcolare quella rilevanza per **coincidenza lessicale**.
- L'**IDF probabilistico di BM25** (`log((N-df+0,5)/(df+0,5))`) può essere **negativo** quando un termine compare in più della metà dei documenti; si corregge tagliando a 0 o con `log(1+...)`.
- **Precisione@k** e **MRR** valutano un motore di ricerca rispetto a una rilevanza fissata **in anticipo**; la precisione@k è limitata da quanti documenti rilevanti ci sono.
- TF-IDF e BM25 sono il "recuperatore" tipico di un sistema **RAG**, ma **falliscono davanti a sinonimi e parafrasi**: è lì che entrano gli embedding densi (lezione 10).

**Compito:** il notebook di pratica (`clase11_practica_busqueda_recuperacion_clasica_IT.ipynb`), con estrazione di parole chiave e riassunto estrattivo. **Prossima:** lezione 12, progetto integratore e ponte verso i transformer.